# Apply the Framework to Your Own Data

Use this notebook when you want to apply the customer-engagement framework to another brand, platform, or industry dataset.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/WindAlan-sw/luxury-brand-customer-engagement.git"
REPO_DIR = Path("luxury-brand-customer-engagement")

# In Colab, clone the repository if the public data folder is not already present.
if not Path("data/public_metrics").exists():
    try:
        if not REPO_DIR.exists():
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_DIR)
    except Exception as e:
        print("Could not clone repository. If the repo is private, upload the repository ZIP or make the repo public before using Colab.")
        print(e)

print("Working directory:", Path.cwd())
print("Public metrics folder exists:", Path("data/public_metrics").exists())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Required minimum schema for your own post-level data:
# brand, retweet_count, reply_count, like_count, quote_count, year, month
example = pd.DataFrame({
    "brand": ["BrandA", "BrandA", "BrandB", "BrandB"],
    "retweet_count": [10, 20, 15, 30],
    "reply_count": [2, 4, 3, 5],
    "like_count": [100, 150, 90, 180],
    "quote_count": [1, 2, 1, 3],
    "year": [2025, 2025, 2025, 2025],
    "month": [1, 2, 1, 2],
})
example

In [ ]:
metric_cols = ["retweet_count", "reply_count", "like_count", "quote_count"]
X = example[metric_cols].astype(float).clip(lower=0) + 1e-12
P = X / X.sum(axis=0)
k = 1 / np.log(len(X))
entropy = -k * (P * np.log(P)).sum(axis=0)
weights = (1 - entropy) / (1 - entropy).sum()
print(weights)

# Normalize metrics and calculate score
Xn = (X - X.min()) / (X.max() - X.min()).replace(0, 1)
example["ce_score"] = Xn.mul(weights, axis=1).sum(axis=1)
display(example)

In [ ]:
summary = example.groupby("brand").agg(
    post_count=("brand", "size"),
    mean_ce_score=("ce_score", "mean"),
    total_likes=("like_count", "sum"),
    total_retweets=("retweet_count", "sum"),
).reset_index()
display(summary)
summary.plot(x="brand", y="mean_ce_score", kind="bar", legend=False)
plt.ylabel("Mean CE score")
plt.tight_layout()
plt.show()